In [1]:
import os
import re # Para extrair o nome do arquivo e limpar
import io # Para ler string como arquivo para pandas
import pandas as pd
import sys

In [19]:

# --- 1. Configurações ---
INPUT_MARKDOWN_FILE = "extracao_all_2.5pr-03.md" # Arquivo gerado pelo script anterior
OUTPUT_EXCEL_FILE = "extracao_all_2.5pr-03.xlsx" # Nome do arquivo Excel final (.xlsx ou .xlsm)

#Colunas esperadas na tabela final (correspondem à pergunta original)
EXPECTED_COLUMNS = [
    "Nome da Disciplina",
    "Carga Horária Teórica",
    "Carga Horária Prática",
    "Carga Horária de Extensão",
    "Carga Horária Total",
    "Descrição (Ementa)",
    "Referências Bibliográficas Básicas",
    "Referências Bibliográficas Complementares"
]

# EXPECTED_COLUMNS = [
#     "1. Nome da Disciplina.",
#     "2. Carga Horária Teórica.",
#     "3. Carga Horária Prática.",
#     "4. Carga Horária de Extensão (se aplicável).",
#     "5. Carga Horária Total.",
#     "6. Descrição (Ementa).",
#     "8. Referências Bibliográficas Complementares.",
#     "9. Referências Bibliográficas Complementares"
# ]
# --- 2. Função para Parsear uma Seção de Markdown ---

def parse_markdown_section(section_text):
    """
    Tenta extrair o nome do arquivo de origem e parsear uma tabela Markdown de uma seção.

    Args:
        section_text (str): O texto de uma seção individual do arquivo Markdown.

    Returns:
        pandas.DataFrame or None: Um DataFrame com os dados da tabela parseada
                                 (incluindo 'Arquivo Origem'), ou None se não houver
                                 tabela válida ou ocorrer um erro.
    """
    # Tentar extrair o nome do arquivo original do cabeçalho ##
    match = re.search(r'^## Resultados para:\s*(.*?)\s*$', section_text, re.MULTILINE)
    if match:
        original_filename = match.group(1).strip()
        print(f"---> Processando seção para: {original_filename}")
    else:
        print("AVISO: Não foi possível extrair o nome do arquivo original desta seção. Pulando.")
        return None

    # Verificar se a seção contém mensagens de erro explícitas ou "Nenhuma disciplina"
    if "***PROMPT BLOQUEADO***" in section_text or \
       "***ERRO:" in section_text or \
       "Nenhuma disciplina correspondente aos critérios foi encontrada" in section_text:
        print(f"     Seção '{original_filename}' contém erro ou nenhuma disciplina. Pulando parsing.")
        return None

    # Tentar encontrar a tabela Markdown (simplificado: procura por |---|)
    if '|' not in section_text or '---|' not in section_text:
         print(f"     Nenhuma tabela Markdown aparente encontrada para '{original_filename}'. Pulando parsing.")
         return None

    # Tentar limpar um pouco o Markdown antes de passar pro Pandas
    # Remover blocos de código ```markdown e ``` (se existirem)
    table_content = re.sub(r'^```markdown\s*|\s*```$', '', section_text, flags=re.MULTILINE | re.DOTALL)
    # Remover o cabeçalho '## Resultados...'
    table_content = re.sub(r'^## Resultados para:.*?\n+', '', table_content, count=1).strip()

    try:
        # Usar pandas para ler a tabela Markdown
        # skiprows=[1] pula a linha '---|---|...'
        # skipinitialspace=True lida com espaços após o '|'
        table_io = io.StringIO(table_content)
        df = pd.read_csv(table_io, sep='|', skiprows=[1], skipinitialspace=True)

        # --- Limpeza do DataFrame ---
        # 1. Remover colunas que são inteiramente vazias (geralmente a primeira e última devido aos pipes)
        df = df.dropna(axis=1, how='all')
        # 2. Remover linhas que são inteiramente vazias (pode acontecer)
        df = df.dropna(axis=0, how='all')
        # 3. Remover a linha de cabeçalho que o pandas leu como dados (primeira linha efetiva)
        #    e resetar o índice se necessário
        if not df.empty:
             # Verificar se a primeira linha parece ser o cabeçalho duplicado (comparando nomes)
             # Isso pode ser frágil. Uma abordagem mais segura é assumir que a primeira linha lida é o cabeçalho.
             # O skiprow=[1] já deve ter pulado o '---'.
             # Se a primeira linha for o header, vamos usá-la como header
             # Vamos limpar os nomes das colunas atuais (que podem ter vindo da linha 0)
             df.columns = [str(col).strip() for col in df.columns]

             # Se a primeira linha *ainda* for o header, vamos removê-la (isso acontece se a IA repetiu o header)
             # Condição heurística: se a primeira linha é igual aos nomes das colunas
             # if list(df.iloc[0].astype(str).str.strip()) == list(df.columns):
             #      df = df.iloc[1:].reset_index(drop=True)

             # 4. Limpar espaços em branco dos nomes das colunas
             df.columns = [col.strip() for col in df.columns]
             # 5. Limpar espaços em branco de todas as células de string
             df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

        # --- Padronização ---
        # 6. Adicionar a coluna de origem
        df['Arquivo Origem'] = original_filename

        # 7. Garantir que todas as colunas esperadas existam, preenchendo com 'NA' se faltarem
        for col in EXPECTED_COLUMNS:
            if col not in df.columns:
                print(f"     AVISO: Coluna '{col}' não encontrada na tabela de '{original_filename}'. Adicionando como 'NA'.")
                df[col] = 'NA'

        # 8. Reordenar as colunas para o padrão desejado + Arquivo Origem
        final_column_order = ['Arquivo Origem'] + EXPECTED_COLUMNS
        # Garantir que apenas colunas existentes sejam selecionadas na ordem
        ordered_df = df[[col for col in final_column_order if col in df.columns]]

        print(f"     Tabela parseada com sucesso para '{original_filename}'. {len(ordered_df)} linha(s) encontradas.")
        return ordered_df

    except pd.errors.EmptyDataError:
        print(f"     AVISO: Tabela em '{original_filename}' parece vazia ou mal formada após limpeza inicial. Pulando parsing.")
        return None
    except pd.errors.ParserError as e:
        print(f"     ERRO de Parsing Pandas em '{original_filename}': {e}. Verifique a formatação da tabela.")
        print(f"     Conteúdo problemático (início):\n{table_content[:300]}...")
        return None
    except Exception as e:
        print(f"     ERRO inesperado ao parsear/limpar tabela de '{original_filename}': {e}")
        print(f"     Conteúdo (início):\n{table_content[:300]}...")
        return None

# --- 3. Leitura e Processamento do Arquivo Markdown ---
print(f"\n--- Lendo o arquivo de resultados brutos: {INPUT_MARKDOWN_FILE} ---")

if not os.path.exists(INPUT_MARKDOWN_FILE):
    print(f"ERRO CRÍTICO: Arquivo de entrada '{INPUT_MARKDOWN_FILE}' não encontrado!")
    sys.exit(1)

all_dataframes = [] # Lista para guardar os DataFrames de cada seção válida
try:
    with open(INPUT_MARKDOWN_FILE, 'r', encoding='utf-8') as f:
        full_content = f.read()

    # Dividir o conteúdo pelas seções (separador '---' entre arquivos)
    # Usar um separador mais robusto que capture a linha --- e espaços ao redor
    sections = re.split(r'\n\s*---\s*\n', full_content)

    if not sections:
        print("ERRO: Não foi possível dividir o arquivo em seções.")
        sys.exit(1)

    # O primeiro elemento pode ser o título geral do arquivo, verificar
    if sections[0].strip().startswith("# Análise Bruta"):
        print("Ignorando título geral do arquivo...")
        sections_to_process = sections[1:]
    else:
        sections_to_process = sections

    print(f"Encontradas {len(sections_to_process)} seções para processar.")

    # Processar cada seção
    for i, section in enumerate(sections_to_process):
        if section.strip(): # Ignorar seções vazias
            print(f"\n--- Processando Seção {i+1}/{len(sections_to_process)} ---")
            df_section = parse_markdown_section(section.strip())
            if df_section is not None and not df_section.empty:
                all_dataframes.append(df_section)
        else:
             print(f"\n--- Ignorando Seção {i+1} (vazia) ---")


except FileNotFoundError:
    print(f"ERRO CRÍTICO: Arquivo de entrada '{INPUT_MARKDOWN_FILE}' não foi encontrado durante a leitura!")
    sys.exit(1)
except Exception as e:
    print(f"ERRO CRÍTICO durante a leitura ou divisão do arquivo '{INPUT_MARKDOWN_FILE}': {e}")
    sys.exit(1)

# --- 4. Consolidação e Limpeza Final ---
print("\n--- Consolidando e Limpando Dados Finais ---")

if not all_dataframes:
    print("Nenhuma tabela válida foi parseada de nenhuma seção.")
    print("Verifique o arquivo de entrada e os logs de erro/aviso acima.")
    sys.exit(0) # Termina normalmente, mas sem gerar arquivo

try:
    # Concatenar todos os DataFrames em um só
    final_df = pd.concat(all_dataframes, ignore_index=True)
    print(f"Total de {len(final_df)} linhas consolidadas.")

    # Limpeza final: Preencher quaisquer NaN restantes com 'NA'
    # Isso garante consistência se alguma célula ficou vazia apesar das tentativas anteriores
    final_df.fillna('NA', inplace=True)

    # Garantir que todas as colunas esperadas + 'Arquivo Origem' existam no DF final
    # (Embora a função de parse já deva ter feito isso, é uma segurança extra)
    final_column_order = ['Arquivo Origem'] + EXPECTED_COLUMNS
    for col in final_column_order:
         if col not in final_df.columns:
              final_df[col] = 'NA' # Adiciona se faltar completamente

    # Reordenar e selecionar apenas as colunas finais desejadas
    final_df = final_df[final_column_order]

    # Remover espaços em branco redundantes que podem ter escapado (segurança extra)
    final_df = final_df.map(lambda x: x.strip() if isinstance(x, str) else x)

    print("Limpeza final concluída.")
    # Opcional: Imprimir as primeiras linhas do DataFrame final para verificação
    # print("\n--- Pré-visualização do DataFrame Final (Primeiras 5 Linhas) ---")
    # print(final_df.head().to_markdown(index=False)) # Imprime em formato Markdown para melhor leitura no console

except Exception as e:
    print(f"ERRO CRÍTICO durante a consolidação ou limpeza final dos DataFrames: {e}")
    print("Isso pode indicar problemas com os dados coletados, mesmo que o parsing inicial tenha parecido funcionar.")
    sys.exit(1)

# --- 5. Salvando no Arquivo Excel ---
print(f"\n--- Salvando dados consolidados em: {OUTPUT_EXCEL_FILE} ---")

try:
    # Salva o DataFrame no arquivo Excel especificado, sem o índice do pandas
    # O engine 'openpyxl' é necessário para .xlsx
    final_df.to_excel(OUTPUT_EXCEL_FILE, index=False, engine='openpyxl')
    print("Arquivo Excel salvo com sucesso!")
    print(f"Localização: {os.path.abspath(OUTPUT_EXCEL_FILE)}")
except ImportError:
    print("\nERRO: A biblioteca 'openpyxl' é necessária para salvar arquivos .xlsx.")
    print("Instale com: pip install openpyxl")
    sys.exit(1)
except Exception as e:
    print(f"ERRO CRÍTICO ao salvar o arquivo Excel '{OUTPUT_EXCEL_FILE}': {e}")
    sys.exit(1)

print("\n--- Processo Concluído ---")


--- Lendo o arquivo de resultados brutos: extracao_all_2.5pr-03.md ---
Ignorando título geral do arquivo...
Encontradas 111 seções para processar.

--- Processando Seção 1/111 ---
---> Processando seção para: 10-ppc_lic_if_toc_palmas.md
     Nenhuma tabela Markdown aparente encontrada para '10-ppc_lic_if_toc_palmas.md'. Pulando parsing.

--- Processando Seção 2/111 ---
---> Processando seção para: 100-ppc_lic_ufvjm.md
     Nenhuma tabela Markdown aparente encontrada para '100-ppc_lic_ufvjm.md'. Pulando parsing.

--- Processando Seção 3/111 ---
---> Processando seção para: 101-ppc_bac_furg.md
     Nenhuma tabela Markdown aparente encontrada para '101-ppc_bac_furg.md'. Pulando parsing.

--- Processando Seção 4/111 ---
---> Processando seção para: 102-ppc_bac_ufpel.md
     Nenhuma tabela Markdown aparente encontrada para '102-ppc_bac_ufpel.md'. Pulando parsing.

--- Processando Seção 5/111 ---
---> Processando seção para: 103-ppc_bac_ufsc.md
     Nenhuma tabela Markdown aparente encontra

SystemExit: 0

C:\Users\pablo\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
